# 激活函数家族（ReLU / GELU / SiLU / SwiGLU）

> Transformer 中 FFN 的非线性变换，选择影响训练稳定性和最终效果。

## 背景
不同激活函数有不同的数学性质和梯度行为：
- **ReLU**：简单高效但有"死神经元"问题
- **GELU**：高斯误差线性单元，BERT/GPT-2 采用，处处可导
- **SiLU**：x × sigmoid(x)，LLaMA 采用，平滑非单调
- **SwiGLU**：SiLU 门控的 GLU 变体，LLaMA FFN 采用

## 公式
$$\text{ReLU}(x) = \max(0, x)$$
$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}[1 + \text{erf}(x/\sqrt{2})]$$
$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$
$$\text{SwiGLU}(x) = \text{SiLU}(W_1 x) \odot W_2 x$$

## 复杂度
- ReLU/SiLU：O(d)，逐元素
- GELU：O(d)，含 erf 但有快速近似
- SwiGLU：O(d × h)，含两个线性层

## 考察点
- ReLU 的死神经元：负区间梯度为 0
- GELU vs SiLU：SiLU 更简单且效果相当
- SwiGLU 的门控机制：SiLU(W1x) 决定 W2x 的通过比例


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReLU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.maximum(x, torch.zeros_like(x))   # 不用 max(x,0.0) 以支持广播/设备

class Sigmoid(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 1.0 / (1.0 + torch.exp(-x))

class Tanh(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.tanh(x)

class SiLU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

class GELU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 0.5 * x * (1.0 + torch.erf(x / torch.sqrt(torch.tensor(2.0))))

In [ ]:
# 与 PyTorch 内置对拍
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
for name, fn, ref in [
    ('ReLU',  ReLU(),  F.relu),
    ('Sigmoid', Sigmoid(), torch.sigmoid),
    ('Tanh',  Tanh(),   torch.tanh),
    ('SiLU',  SiLU(),   F.silu),
    ('GELU',  GELU(),   F.gelu),
]:
    out = fn(x)
    print(f'{name:8s}: {out.tolist()}  match={torch.allclose(out, ref(x))}')

In [ ]:
# 导数验证（autograd vs 解析）
x = torch.randn(5, requires_grad=True)
s = torch.sigmoid(x)
s.sum().backward()
print('sigmoid grad autograd:', x.grad.tolist())
print('sigmoid grad analytic:', (torch.sigmoid(x.detach()) * (1 - torch.sigmoid(x.detach()))).tolist())

## 小结 / 易错点
- `torch.max(x, 0.0)` 在新版本会广播报错，用 `torch.maximum(x, zeros_like(x))` 更稳。
- Sigmoid 导数用前向值 $s(1-s)$，不要重算 exp。
- SiLU 非单调，负区不为 0，这是它优于 ReLU 的关键。
- GELU 精确式用 `erf`，近似式用 tanh 展开（推理常融合）。

## ✅ 测试验证

In [ ]:
# 验证激活函数性质
import torch
import torch.nn.functional as F

x = torch.randn(10)

# SiLU (Swish): x * sigmoid(x)
silu = x * torch.sigmoid(x)
assert torch.allclose(silu, F.silu(x), atol=1e-6), "SiLU mismatch"
print("  ✓ SiLU = x * sigmoid(x)")

# GELU: x * 0.5 * (1 + erf(x / sqrt(2)))
gelu = x * 0.5 * (1 + torch.erf(x / (2 ** 0.5)))
assert torch.allclose(gelu, F.gelu(x), atol=1e-6), "GELU mismatch"
print("  ✓ GELU = x * 0.5 * (1 + erf(x/√2))")

# ReLU: max(0, x)
relu = torch.clamp(x, min=0)
assert torch.allclose(relu, F.relu(x), atol=1e-6), "ReLU mismatch"
print("  ✓ ReLU = max(0, x)")

# SiLU 在 x=0 处为 0
assert abs((0 * torch.sigmoid(torch.tensor(0.0))).item()) < 1e-6
print("  ✓ SiLU(0) = 0")

# GELU 在 x=0 处为 0
assert abs(F.gelu(torch.tensor(0.0)).item()) < 1e-6
print("  ✓ GELU(0) = 0")

print("✅ ActivationFunction 测试通过: SiLU/GELU/ReLU 与 PyTorch 一致")
